In [1]:
import glob
import os
import pandas as pd

root = "data/raw/crisislex/CrisisLexT26"
files = [f for f in glob.glob(f"{root}/**/*", recursive=True)
         if os.path.isfile(f)
         and "__MACOSX" not in f
         and not os.path.basename(f).startswith("._")]
print(len(files), "files")
for f in files:
    print(f, os.path.getsize(f))

csvs = [f for f in files if f.endswith(".csv")]
print(len(csvs), "csv files")
first = pd.read_csv(csvs[0], encoding_errors="replace", on_bad_lines="skip")
print(csvs[0])
print(first.shape)
print(list(first.columns))
first.head()

105 files
data/raw/crisislex/CrisisLexT26\README.md 750
data/raw/crisislex/CrisisLexT26\2012_Colorado_wildfires\2012_Colorado_wildfires-event_description.json 1114
data/raw/crisislex/CrisisLexT26\2012_Colorado_wildfires\2012_Colorado_wildfires-tweetids_entire_period.csv 234227
data/raw/crisislex/CrisisLexT26\2012_Colorado_wildfires\2012_Colorado_wildfires-tweets_labeled.csv 224671
data/raw/crisislex/CrisisLexT26\2012_Colorado_wildfires\README.md 2893
data/raw/crisislex/CrisisLexT26\2012_Costa_Rica_earthquake\2012_Costa_Rica_earthquake-event_description.json 891
data/raw/crisislex/CrisisLexT26\2012_Costa_Rica_earthquake\2012_Costa_Rica_earthquake-tweetids_entire_period.csv 132699
data/raw/crisislex/CrisisLexT26\2012_Costa_Rica_earthquake\2012_Costa_Rica_earthquake-tweets_labeled.csv 258836
data/raw/crisislex/CrisisLexT26\2012_Costa_Rica_earthquake\README.md 2896
data/raw/crisislex/CrisisLexT26\2012_Guatemala_earthquake\2012_Guatemala_earthquake-event_description.json 890
data/raw/crisis

,Timestamp,Tweet-ID,Included(Y/N)
0,Fri Jun 08 06:22:06 +0000 2012,210980000776728576,Y
1,Fri Jun 08 07:42:10 +0000 2012,211000150196367360,Y
2,Fri Jun 08 10:23:20 +0000 2012,211040709124440064,Y
3,Fri Jun 08 12:01:32 +0000 2012,211065421984571393,Y
4,Fri Jun 08 12:40:41 +0000 2012,211075274413056000,Y


In [2]:
import os
import pandas as pd

root = "data/raw/crisislex/CrisisLexT26"
floods = ["2012_Philipinnes_floods", "2013_Alberta_floods", "2013_Colorado_floods",
          "2013_Manila_floods", "2013_Queensland_floods", "2013_Sardinia_floods"]

print(open(os.path.join(root, "README.md"), encoding="utf-8", errors="replace").read())
print("=" * 60)
print(open(os.path.join(root, floods[1], "README.md"), encoding="utf-8", errors="replace").read())

frames = []
for ev in floods:
    p = os.path.join(root, ev, f"{ev}-tweets_labeled.csv")
    d = pd.read_csv(p, encoding_errors="replace", on_bad_lines="skip")
    d.columns = [c.strip() for c in d.columns]
    d["event"] = ev
    frames.append(d)

t26 = pd.concat(frames, ignore_index=True)
print(t26.shape)
print(list(t26.columns))
print(t26["event"].value_counts())

for c in t26.columns:
    if t26[c].nunique() <= 15:
        print("\n", c)
        print(t26[c].value_counts())

t26.head(5)

Contents of this directory

This directory contains tweets posted during a series of crisis events. See the README.md files inside each directory for details.

Reference
---------
[Olteanu et al. 2015] Alexandra Olteanu, Sarah Vieweg, Carlos Castillo. 2015. What to Expect When the Unexpected Happens: Social Media Communications Across Crises. In Proceedings of the ACM 2015 Conference on Computer Supported Cooperative Work and Social Computing (CSCW '15). ACM, Vancouver, BC, Canada.

Questions/inquiries
-------------------
For inquiries please contact [Alexandra Olteanu](mailto:alexandra.olteanu@epfl.ch), or Sarah Vieweg, or Carlos Castillo.
 
Version history
---------------

 * 2014-10-26: v1.0, initial release.


Contents of this directory
This directory contains tweets posted during **2013 Alberta floods** organized in two comma-separated values (.csv) files containing tweet-ids for all the tweets filtered for this event, plus the text of the tweets and labels only for the labeled on

,Tweet ID,Tweet Text,Information Source,Information Type,Informativeness,event
0,230634706436227073,Philippine floods worsen as death toll hits 14...,Media,Affected individuals,Related and informative,2012_Philipinnes_floods
1,230692092911964161,RT @gmanews: Philippine floods worsen as death...,Media,Affected individuals,Related and informative,2012_Philipinnes_floods
2,230882841498628096,Philippine flood fatalities hit 23 http://t.co...,Media,Other Useful Information,Related and informative,2012_Philipinnes_floods
3,232034974851805187,4 Luzon dams release water; flood warning up: ...,Media,Caution and advice,Related and informative,2012_Philipinnes_floods
4,232098107490439169,RT @ANCALERTS: PAGASA 8:50pm advisory: Yellow ...,Media,Caution and advice,Related and informative,2012_Philipinnes_floods


In [3]:
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

NON_HAZARD_TYPES = ["Sympathy and support", "Donations and volunteering", "Other Useful Information"]

def to_safety(row):
    if row["Information Type"] == "Infrastructure and utilities":
        return "hazard"
    if row["Information Type"] in NON_HAZARD_TYPES:
        return "non_hazard"
    if row["Informativeness"] == "Not related":
        return "non_hazard"
    return None

def clean_tweet(t):
    t = str(t)
    t = re.sub(r"https?://\S+", " ", t)
    t = re.sub(r"@\w+", " ", t)
    t = re.sub(r"^rt\s+", "", t, flags=re.I)
    t = t.replace("#", " ")
    return re.sub(r"\s+", " ", t).strip()

t26["safety_label"] = t26.apply(to_safety, axis=1)
t26s = t26.dropna(subset=["safety_label"]).copy()
t26s["text"] = t26s["Tweet Text"].apply(clean_tweet)
t26s = t26s[t26s["text"].str.len() > 15].drop_duplicates(subset="text")
print(t26s.shape)
print(t26s["safety_label"].value_counts())
print(t26s.groupby(["event", "safety_label"]).size().unstack(fill_value=0))

(3881, 8)
safety_label
non_hazard    3268
hazard         613
Name: count, dtype: int64
safety_label             hazard  non_hazard
event                                      
2012_Philipinnes_floods      57         498
2013_Alberta_floods         190         525
2013_Colorado_floods        166         487
2013_Manila_floods           29         494
2013_Queensland_floods      119         654
2013_Sardinia_floods         52         610


In [4]:
pd.set_option("display.max_colwidth", 160)
print("--- hazard examples ---")
display(t26s[t26s["safety_label"] == "hazard"].sample(6, random_state=1)[["text", "event"]])
print("--- Sardinia sample (language check) ---")
display(t26s[t26s["event"] == "2013_Sardinia_floods"].sample(5, random_state=1)[["text"]])

--- hazard examples ---


,text,event
2942,: Fracking and Flooding in Colorado: The More We Know the Worse It Gets,2013_Colorado_floods
6014,Tracimazione alla diga di torpè Sardegna,2013_Sardinia_floods
1197,: A google map showing yyc evacuated area's. Extensive - much downtown. yycflood evacuated community yycre,2013_Alberta_floods
4226,YIKES!! Tree + Powerlines!! Toowoomba bigwet,2013_Queensland_floods
5728,: Emergenza Sardegna: pensiamo alla ricostruzione - Rete del Dono ForzaSardegna,2013_Sardinia_floods
2861,Colorado flooding puts oil and gas wells underwater,2013_Colorado_floods


--- Sardinia sample (language check) ---


,text
5731,": RT : “ : Rai VERGOGNA Sardegna, tutta da leggere."
6196,: Golfo degli Angeli dalla Sella del Diavolo. cagliari Sardegna
5280,: La sardegna è in ginocchio: date notizie da e per quella terra!
5392,“Ciclone? Uragano? Nubifragio? Forse certi giornalisti avrebbero bisogno di seguire i rudimenti di meteorologia allertameteoSAR Sardegna
5638,Hit Chart è vicina alle popolazioni della Sardegna colpite dall'alluvione. Vi ricordiamo alcuni numeri utili. -...


In [5]:
sf = pd.read_csv("data/processed/humaid_safety.csv")
HOLDOUT = ["maryland_floods_2018", "srilanka_floods_2017"]
train = sf[(~sf["event"].isin(HOLDOUT)) & (sf["split"] == "train")]

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, lowercase=True)
clf = LogisticRegression(max_iter=1000, class_weight="balanced").fit(vec.fit_transform(train["text"]), train["safety_label"])
HZ = list(clf.classes_).index("hazard")
print("trained on", len(train), "HumAID tweets")

trained on 16986 HumAID tweets


In [6]:
EXCLUDE = ["2013_Sardinia_floods"]   # e.g.  the language check shows mostly Italian
test = t26s[~t26s["event"].isin(EXCLUDE)]
p = clf.predict_proba(vec.transform(test["text"]))[:, HZ]

for th in [0.5, 0.35]:
    pred = np.where(p >= th, "hazard", "non_hazard")
    print(f"\n=== T26 floods, threshold {th} ===")
    print(classification_report(test["safety_label"], pred))

pred = p >= 0.35
y = (test["safety_label"] == "hazard").values
print("\nPer event at threshold 0.35:")
for ev in test["event"].unique():
    m = (test["event"] == ev).values
    tp = int((pred[m] & y[m]).sum())
    print(f"{ev:26s} n={m.sum():4d} hazard={int(y[m].sum()):3d} "
          f"recall={tp/max(int(y[m].sum()),1):.2f} precision={tp/max(int(pred[m].sum()),1):.2f}")


=== T26 floods, threshold 0.5 ===
              precision    recall  f1-score   support

      hazard       0.71      0.21      0.33       561
  non_hazard       0.86      0.98      0.91      2658

    accuracy                           0.85      3219
   macro avg       0.78      0.60      0.62      3219
weighted avg       0.83      0.85      0.81      3219


=== T26 floods, threshold 0.35 ===
              precision    recall  f1-score   support

      hazard       0.54      0.44      0.49       561
  non_hazard       0.89      0.92      0.90      2658

    accuracy                           0.84      3219
   macro avg       0.71      0.68      0.70      3219
weighted avg       0.83      0.84      0.83      3219


Per event at threshold 0.35:
2012_Philipinnes_floods    n= 555 hazard= 57 recall=0.09 precision=0.23
2013_Alberta_floods        n= 715 hazard=190 recall=0.44 precision=0.65
2013_Colorado_floods       n= 653 hazard=166 recall=0.53 precision=0.62
2013_Manila_floods         n=

In [7]:
pd.set_option("display.max_colwidth", 140)
test = test.copy()
test["p"] = p
missed = test[(test["safety_label"] == "hazard") & (test["p"] < 0.35)]
print("MISSED hazards (model said non_hazard):", len(missed))
display(missed[missed["event"].isin(["2013_Alberta_floods", "2013_Queensland_floods"])]
        .sample(10, random_state=1)[["text", "p"]])
display(missed[missed["event"].isin(["2012_Philipinnes_floods", "2013_Manila_floods"])]
        .sample(8, random_state=1)[["text", "p"]])

MISSED hazards (model said non_hazard): 313


,text,p
1298,: yycflood East Elbow Park,0.245150
1321,": Can you see the car? RT : PHOTO: Macleod Trail under water, courtesy of Jeff Berg. YYC yycflood",0.249828
4136,: Warrego Alert: Truckies reporting Highway east of Dalby starting to fall apart. bigwet qldfloods,0.177120
1642,: WHOA!?! Heavy erosion UNDER the pavement RT ‏ 25th Ave. Erlton/Mission yycflood,0.304148
5132,All Brisbane catholic schools open today except for St Joseph's School (Gayndah) St Mary's School (Laidley) bigwet qldfloods,0.259541
1120,: Alberta gas leak continues as extreme flooding prevents shutdown,0.217483
4212,Rain coming under closed front door now. bigwet brisbane,0.292175
1224,: . beltline 13 yycflood . inundation in Beltline Victoria Park .,0.210092
4968,": Flooding in Maryborough, Australia - visible as we flew over Tuesday morning, 29 Jan.",0.243192
1142,: 2nd Ave. yycflood,0.243666


,text,p
331,!!!!! RT PLEASE RT!! UST HOSPITAL'S GENERATOR ALREADY DIED!!! PLEASE SEND SOME RESCUERS! RescuePH ReliefPH,0.092633
3183,Who's in the FEU-UST area? How's the flood there? MaringPH,0.188521
277,: RT : UST Hospital is running out of food. They need help. RescuePH,0.062931
3708,Manila City Hall-Lawton-Quiapo-Recto rain flood typhoon typhoonmaring warning,0.244234
3179,: RT Rep. : Tirona Hiway near SM Bacoor Passable only for buses.Not safe for light vehicles. MaringPH,0.199320
368,PGH is also in need of help. ReliefPH,0.112140
755,: FloodsPH: Cubao underpass now closed to traffic. | via,0.233757
426,"Pls help the people here in TUA High School along E. Rod they need food, water, clothes and blankets! RescuePH",0.043295


## Findings

- CrisisLexT26 flood events (Sardinia excluded: Italian): 3,219 labelled tweets, 561 hazard ("Infrastructure and utilities").
- HumAID-trained TF-IDF + logistic regression: hazard recall 0.21 (threshold 0.50) and 0.44 (threshold 0.35); precision 0.71 / 0.54.
- Accuracy 84–85% is barely above the naive 82.6% ("always non_hazard"), so recall/F1 is the meaningful measure.
- Transfer is weak across datasets: labelling guidelines, years (2012–13 vs 2016–19) and possibly language differ. Philippines/Manila events are worst (recall 0.09 / 0.24).
- Threshold was chosen on the HumAID dev split and not re-tuned on this test set.
- Conclusion: baseline is only reliable in-domain; a fine-tuned language model on in-domain NYC data is needed.
- - Missed hazards (313 of 561): a mix of (a) real short eyewitness reports about roads, underpasses, highways and power failures that the model scores low (0.04–0.30), and (b) label disagreement: T26 "Infrastructure and utilities" also contains help requests and photo-only tweets that HumAID would label differently.
- Philippines/Manila tweets are mostly English with Filipino hashtags, so language is not the main cause there.
- Event-specific hashtags (yycflood, qldfloods) are unseen tokens for a HumAID-trained model.
- Candidate fixes: train on HumAID + T26 and test on whole held-out events; add infrastructure keywords (road, highway, underpass, bridge, generator, power) as features or rules; fine-tune a language model.